All the libraries

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras import layers, models
import tensorflow as tf
import tensorflow_datasets as tfds
from skimage.color import rgb2lab, lab2rgb
import numpy as np
import math
from google.colab import drive
import os
import tensorflow as tf
from tensorflow.keras import layers as KL, models
import torch

GPU

In [2]:
gpu_avail = torch.cuda.is_available()
print(f"Is the GPU available? {gpu_avail}")

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print(f"Device: {device}")

Is the GPU available? True
Device: cuda


Some parameters

In [7]:
# === CONFIGURATION ===
IMG_SIZE = 128
BATCH_SIZE = 32

Preprocess image

In [3]:
def preprocess_image(sample):
    """Resize and normalize image"""
    image = tf.image.resize(sample['image'], (IMG_SIZE, IMG_SIZE))
    image = tf.cast(image, tf.float32) / 255.0
    return image

RGB to LAB and Viceversa

In [20]:
import numpy as np
import warnings
from skimage.color import lab2rgb, rgb2lab

def lab_batch_to_rgb(L_batch01, ab_batch_norm, clip_lab=True, suppress_warnings=True):
    """
    Converte batch di (L in [0,1], ab in [-1,1]) -> RGB in [0,1].
    L_batch01: (B,H,W,1) float, [0,1]
    ab_batch_norm: (B,H,W,2) float, [-1,1]
    Ritorna: rgb (B,H,W,3) float32, [0,1]
    """
    L_batch01 = np.asarray(L_batch01)
    ab_batch_norm = np.asarray(ab_batch_norm)
    assert L_batch01.ndim == 4 and L_batch01.shape[-1] == 1, "L_batch01 deve essere (B,H,W,1)"
    assert ab_batch_norm.ndim == 4 and ab_batch_norm.shape[-1] == 2, "ab_batch_norm deve essere (B,H,W,2)"
    assert L_batch01.shape[:3] == ab_batch_norm.shape[:3], "L e ab devono avere stessa (B,H,W)"

    B, H, W, _ = L_batch01.shape

    # 1) Denormalizza a range Lab canonico
    L = (L_batch01 * 100.0).astype(np.float64)            # [0,100]
    ab = (ab_batch_norm * 128.0).astype(np.float64)       # circa [-128,128]

    # 2) Clip leggero per stabilità (evita bordi problematici)
    if clip_lab:
        L  = np.clip(L,  0.0 + 1e-6, 100.0 - 1e-6)
        ab = np.clip(ab, -127.0,      127.0)

    lab = np.concatenate([L, ab], axis=-1)                # (B,H,W,3)

    # 3) lab2rgb immagine-per-immagine
    rgb_list = []
    ctx = warnings.catch_warnings()
    if suppress_warnings:
        warnings.simplefilter("ignore")
    with ctx:
        for i in range(B):
            rgb = lab2rgb(lab[i].astype(np.float64))      # (H,W,3) in [0,1] (può leggermente uscire)
            rgb_list.append(rgb)

    rgb = np.stack(rgb_list, axis=0).astype(np.float32)
    rgb = np.clip(rgb, 0.0, 1.0)                          # forza [0,1] pulito
    return rgb


def rgb_batch_to_lab(rgb_batch01, clip_rgb=True, normalize_ab=True, return_L01=True, suppress_warnings=True):
    """
    Converte batch di RGB [0,1] -> (L,ab) in formato del tuo modello.
    rgb_batch01: (B,H,W,3) float, atteso in [0,1]
    Ritorna (L_batch01, ab_batch_norm) se return_L01 & normalize_ab,
            altrimenti ritorna L* in [0,100] e/o a*,b* non normalizzati.
    """
    rgb_batch01 = np.asarray(rgb_batch01)
    assert rgb_batch01.ndim == 4 and rgb_batch01.shape[-1] == 3, "rgb_batch01 deve essere (B,H,W,3)"

    if clip_rgb:
        rgb = np.clip(rgb_batch01, 0.0, 1.0).astype(np.float64)
    else:
        rgb = rgb_batch01.astype(np.float64)

    B = rgb.shape[0]
    lab_list = []
    ctx = warnings.catch_warnings()
    if suppress_warnings:
        warnings.simplefilter("ignore")
    with ctx:
        for i in range(B):
            lab_img = rgb2lab(rgb[i])                     # (H,W,3): L* in [0,100], a*,b* ~ [-128,128]
            lab_list.append(lab_img)

    lab = np.stack(lab_list, axis=0)                      # (B,H,W,3)
    L_star = lab[..., :1]                                 # (B,H,W,1)
    ab     = lab[..., 1:]                                 # (B,H,W,2)

    # Normalizzazioni per tornare al tuo formato
    if return_L01:
        L_batch01 = (L_star / 100.0).astype(np.float32)   # [0,1]
    else:
        L_batch01 = L_star.astype(np.float32)             # [0,100]

    if normalize_ab:
        ab_batch_norm = (ab / 128.0).astype(np.float32)   # circa [-1,1]
        # opzionale un piccolo clip per sicurezza numerica:
        ab_batch_norm = np.clip(ab_batch_norm, -1.1, 1.1)
    else:
        ab_batch_norm = ab.astype(np.float32)             # ~[-128,128]

    return L_batch01, ab_batch_norm


Generator for grey

In [5]:
# === DATA GENERATOR ===
class GrayScaleGenerator(tf.keras.utils.Sequence):
    def __init__(self, images, batch_size=BATCH_SIZE, shuffle=True):
        self.images = images.numpy() # Convert to numpy array
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.indices = np.arange(len(images))
        if self.shuffle:
            np.random.shuffle(self.indices)

    def __len__(self):
        return len(self.images) // self.batch_size

    def __getitem__(self, idx):
        batch_indices = self.indices[idx * self.batch_size:(idx + 1) * self.batch_size]
        batch_color = self.images[batch_indices]
        batch_gray = np.mean(batch_color,axis=-1)
        batch_palette = np.mean(batch_color,axis=(1,2)) # batch of color palettes
        return (batch_gray, batch_palette), batch_color

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)

NameError: name 'BATCH_SIZE' is not defined

Generator for L + palette

In [8]:

class LabPaletteGenerator(tf.keras.utils.Sequence):
    """
    Prende immagini RGB in [0,1] e produce:
      inputs  = (L, palette)   -> L:(B,H,W,1) [0,1], palette:(B,3) [0,1]
      targets = ab             -> (B,H,W,2)   [-1,1]
    """
    def __init__(self, images, batch_size=BATCH_SIZE, shuffle=True):
        self.images  = images.numpy() if tf.is_tensor(images) else images
        self.images  = self.images.astype(np.float32)
        self.batch_size = int(batch_size)
        self.shuffle = bool(shuffle)
        self.indices = np.arange(len(self.images))
        if self.shuffle:
            np.random.shuffle(self.indices)

    def __len__(self):
        # usa ceil per non perdere l'ultimo spezzone
        return int(math.ceil(len(self.images) / self.batch_size))

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)

    def __getitem__(self, idx):
        s = idx * self.batch_size
        e = min((idx + 1) * self.batch_size, len(self.images))
        batch_idx = self.indices[s:e]

        batch_color  = self.images[batch_idx]                          # (B,H,W,3) in [0,1]
        batch_palette = batch_color.mean(axis=(1,2)).astype(np.float32) # (B,3) in [0,1]

        L, ab = rgb_batch_to_lab(batch_color)  # L:(B,H,W,1), ab:(B,H,W,2)
        return ( (L, batch_palette), ab )


Loaders and splitters as the prof proposes

In [9]:
# === LOAD & SPLIT FLOWER DATA ===
def load_flower_dataset_splits():
    ds_train = tfds.load('oxford_flowers102', split='test', shuffle_files=True)
    ds_val = tfds.load('oxford_flowers102', split='validation', shuffle_files=True)
    ds_test = tfds.load('oxford_flowers102', split='train', shuffle_files=True)
    ds_train = tf.stack(list(ds_train.map(preprocess_image)))
    ds_val = tf.stack(list(ds_val.map(preprocess_image)))
    ds_test = tf.stack(list(ds_test.map(preprocess_image)))
    return (ds_train,ds_val,ds_test)

Load and splitters extended to use all the dataset

In [10]:
def load_flower_dataset_splits():
    base = "train+validation+test"  # unisci tutto

    ds_train = tfds.load(
        'oxford_flowers102',
        split=f'{base}[:80%]',
        shuffle_files=True,
        read_config=tfds.ReadConfig(shuffle_seed=42)
    )
    ds_val = tfds.load(
        'oxford_flowers102',
        split=f'{base}[80%:90%]',
        shuffle_files=True,
        read_config=tfds.ReadConfig(shuffle_seed=42)
    )
    ds_test = tfds.load(
        'oxford_flowers102',
        split=f'{base}[90%:]',
        shuffle_files=True,
        read_config=tfds.ReadConfig(shuffle_seed=42)
    )

    ds_train = tf.stack(list(ds_train.map(preprocess_image)))
    ds_val   = tf.stack(list(ds_val.map(preprocess_image)))
    ds_test  = tf.stack(list(ds_test.map(preprocess_image)))

     # Stampe utili
    print("[Flowers102 80/10/10]")
    print(f"  train: {ds_train.shape[0]} immagini")
    print(f"  val:   {ds_val.shape[0]} immagini")
    print(f"  test:  {ds_test.shape[0]} immagini")
    print(f"  image shape per sample: {ds_train.shape[1:]}  dtype={ds_train.dtype}")

    return (ds_train, ds_val, ds_test)


TO use the grayscale

In [ ]:
train_imgs, val_imgs, test_imgs = load_flower_dataset_splits()

train_gen = GrayScaleGenerator(train_imgs, batch_size=BATCH_SIZE)
val_gen = GrayScaleGenerator(val_imgs, batch_size=BATCH_SIZE)
test_gen = GrayScaleGenerator(test_imgs, batch_size=BATCH_SIZE)

To use the L + palette

In [11]:
train_imgs, val_imgs, test_imgs = load_flower_dataset_splits()

train_lab = LabPaletteGenerator(train_imgs, batch_size=BATCH_SIZE, shuffle=True)
val_lab   = LabPaletteGenerator(val_imgs,   batch_size=BATCH_SIZE, shuffle=False)
test_lab  = LabPaletteGenerator(test_imgs,  batch_size=BATCH_SIZE, shuffle=False)


Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Extraction completed...: 0 file [00:00, ? file/s]

Generating splits...:   0%|          | 0/3 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

...

Generating test examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/oxford_flowers102/incomplete.2U61NM_2.1.1/oxford_flowers102-test.tfrecord*…

Generating validation examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/oxford_flowers102/incomplete.2U61NM_2.1.1/oxford_flowers102-validation.tfr…

Dataset oxford_flowers102 downloaded and prepared to /root/tensorflow_datasets/oxford_flowers102/2.1.1. Subsequent calls will reuse this data.
[Flowers102 80/10/10]
  train: 6959 immagini
  val:   2655 immagini
  test:  2655 immagini
  image shape per sample: (128, 128, 3)  dtype=<dtype: 'float32'>


MODEL - Unet + FiLM

In [12]:
# ---------- FiLM ----------
def film_mod(x, palette, channels):
    gamma = KL.Dense(channels, kernel_initializer="zeros", bias_initializer="zeros")(palette)
    beta  = KL.Dense(channels, kernel_initializer="zeros", bias_initializer="zeros")(palette)
    g = KL.Reshape((1,1,channels))(gamma)
    b = KL.Reshape((1,1,channels))(beta)
    scale = KL.Lambda(lambda t: t + 1.0)(g)   # x * (1+g) + b
    x = KL.Multiply()([x, scale])
    x = KL.Add()([x, b])
    return x

# ---------- Squeeze-Excitation ----------
def se_block(x, r=16):  # r=16 (più leggero di r=8)
    c = x.shape[-1]
    s = KL.GlobalAveragePooling2D()(x)
    s = KL.Dense(max(c//r, 4), activation='relu')(s)
    s = KL.Dense(c, activation='sigmoid')(s)
    s = KL.Reshape((1,1,c))(s)
    return KL.Multiply()([x, s])

# ---------- Residual Conv Block + FiLM (+SE opzionale) ----------
def res_film_block(x, c, palette, sep=False, use_se=True, se_r=16):
    Conv = KL.SeparableConv2D if sep else KL.Conv2D
    x_in = x
    x = Conv(c, 3, padding="same")(x); x = KL.BatchNormalization()(x); x = KL.Activation('swish')(x)
    x = film_mod(x, palette, c)
    x = Conv(c, 3, padding="same")(x); x = KL.BatchNormalization()(x)
    if use_se:
        x = se_block(x, r=se_r)
    if x_in.shape[-1] != c:
        x_in = KL.Conv2D(c, 1, padding="same")(x_in); x_in = KL.BatchNormalization()(x_in)
    x = KL.Add()([x, x_in]); x = KL.Activation('swish')(x)
    return x

def build_unet_film_max(img_size=128, base=48, sep=False, se=True, se_r=16, dilate=False):
    inp_L = KL.Input((img_size, img_size, 1), name="L_input")
    inp_p = KL.Input((3,), name="palette_input")

    # Encoder
    c1 = res_film_block(inp_L, base,     inp_p, sep, se, se_r); p1 = KL.MaxPool2D()(c1)
    c2 = res_film_block(p1,    base*2,   inp_p, sep, se, se_r); p2 = KL.MaxPool2D()(c2)
    c3 = res_film_block(p2,    base*4,   inp_p, sep, se, se_r); p3 = KL.MaxPool2D()(c3)

    # Bottleneck
    b  = res_film_block(p3,    base*8,   inp_p, sep, se, se_r)
    if dilate:
        bd = KL.Conv2D(base*8, 3, padding="same", dilation_rate=2)(b)
        bd = KL.BatchNormalization()(bd); bd = KL.Activation('swish')(bd)
        bd = film_mod(bd, inp_p, base*8)
        b  = KL.Add()([b, bd])

    # Decoder
    u3 = KL.UpSampling2D()(b);  u3 = KL.Concatenate()([u3, c3]); u3 = res_film_block(u3, base*4, inp_p, sep, se, se_r)
    u2 = KL.UpSampling2D()(u3); u2 = KL.Concatenate()([u2, c2]); u2 = res_film_block(u2, base*2, inp_p, sep, se, se_r)
    u1 = KL.UpSampling2D()(u2); u1 = KL.Concatenate()([u1, c1]); u1 = res_film_block(u1, base,   inp_p, sep, se, se_r)

    out_ab = KL.Conv2D(2, 1, activation="tanh", name="ab_out")(u1)
    return models.Model([inp_L, inp_p], out_ab, name="UNet_FiLM_max")


In [16]:
tf.keras.backend.clear_session()
model = build_unet_film_max(img_size=IMG_SIZE, base=48, sep=False, se=True, se_r=16, dilate=False)
model.summary(); print("Params:", model.count_params())



Model: "UNet_FiLM_max"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ L_input             │ (None, 128, 128,  │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ palette_input       │ (None, 3)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 128, 128,  │        480 │ L_input[0][0]     │
│                     │ 48)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 48)        │        192 │ palette_input[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 128, 128,  │        192 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 48)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape (Reshape)   │ (None, 1, 1, 48)  │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 128, 128,  │          0 │ batch_normalizat… │
│ (Activation)        │ 48)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda (Lambda)     │ (None, 1, 1, 48)  │          0 │ reshape[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 48)        │        192 │ palette_input[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply (Multiply) │ (None, 128, 128,  │          0 │ activation[0][0], │
│                     │ 48)               │            │ lambda[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_1 (Reshape) │ (None, 1, 1, 48)  │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 128, 128,  │          0 │ multiply[0][0],   │
│                     │ 48)               │            │ reshape_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 128, 128,  │     20,784 │ add[0][0]         │
│                     │ 48)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128, 128,  │        192 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 48)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 48)        │          0 │ batch_normalizat… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 4)         │        196 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 48)        │        240 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_2 (Reshape) │ (None, 1, 1, 48)  │          0 │ dense_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 128, 128,  │         96 │ L_input[0][0]     │
│                     │ 48)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 4,673,926 (17.83 MB)

 Trainable params: 4,667,590 (17.81 MB)

 Non-trainable params: 6,336 (24.75 KB)

Params: 4673926


Loss function - MSA + (very small) TV

In [13]:
def mae_ab_loss(y_true, y_pred):
    # y_* shape: (B,H,W,2) in [-1,1]
    return tf.reduce_mean(tf.abs(y_pred - y_true))

def mae_tv_loss(lambda_tv=1e-4):
    def loss(y_true, y_pred):
        mae = tf.reduce_mean(tf.abs(y_pred - y_true))
        tv  = tf.reduce_mean(tf.image.total_variation(y_pred))  # su (a,b)
        return mae + lambda_tv * tv
    return loss

def mse_cos_on_ab(alpha=1.0, beta=0.1, eps=1e-6):
    def loss(y_true, y_pred):
        mse = tf.reduce_mean(tf.square(y_pred - y_true))
        t = tf.nn.l2_normalize(y_true, axis=-1, epsilon=eps)
        p = tf.nn.l2_normalize(y_pred, axis=-1, epsilon=eps)
        cos = tf.reduce_mean(tf.reduce_sum(t * p, axis=-1))  # ∈[-1,1]
        return alpha * mse + beta * (1.0 - cos)
    return loss

Google drive connectivity

In [14]:
# 1) Monta Google Drive
drive.mount('/content/drive')

# 2) Scegli una cartella su Drive dove salvare i pesi
SAVE_DIR = '/content/drive/MyDrive/Colorizer'   # cambia nome/cartella se vuoi
os.makedirs(SAVE_DIR, exist_ok=True)

# 3) Path per il checkpoint
CKPT_PATH = os.path.join(SAVE_DIR, 'best_colorizer_film.weights.h5')

Mounted at /content/drive


Compile and training

In [17]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(3e-4),
    loss=mae_tv_loss(lambda_tv=1e-4),          # MAE + TV è la loss ottimizzata
    metrics=[tf.keras.metrics.MeanAbsoluteError(name='mae'),
             tf.keras.metrics.MeanSquaredError(name='mse')]
)


In [18]:
callbacks = [
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',    # MINIMIZZIAMO la loss
        patience=3,
        factor=0.5,
        min_lr=1e-6,
        verbose=1,
        mode='min'
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=8,
        restore_best_weights=True,
        verbose=1,
        mode='min'
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=CKPT_PATH,
        monitor='val_loss',
        save_best_only=True,
        save_weights_only=True,
        mode='min',
        verbose=1
    )
]


In [22]:
history = model.fit(
    train_lab,
    validation_data=val_lab,
    epochs=100,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/100
218/218 ━━━━━━━━━━━━━━━━━━━━ 0s 110ms/step - loss: 0.1280 - mae: 0.1098 - mse: 0.0246
Epoch 1: val_loss did not improve from 0.12861
218/218 ━━━━━━━━━━━━━━━━━━━━ 33s 152ms/step - loss: 0.1279 - mae: 0.1098 - mse: 0.0246 - val_loss: 0.1469 - val_mae: 0.1237 - val_mse: 0.0287 - learning_rate: 3.0000e-04
Epoch 2/100
218/218 ━━━━━━━━━━━━━━━━━━━━ 0s 109ms/step - loss: 0.1192 - mae: 0.1028 - mse: 0.0211
Epoch 2: val_loss improved from 0.12861 to 0.12181, saving model to /content/drive/MyDrive/Colorizer/best_colorizer_film.weights.h5
218/218 ━━━━━━━━━━━━━━━━━━━━ 35s 159ms/step - loss: 0.1191 - mae: 0.1028 - mse: 0.0211 - val_loss: 0.1218 - val_mae: 0.1039 - val_mse: 0.0203 - learning_rate: 3.0000e-04
Epoch 3/100
218/218 ━━━━━━━━━━━━━━━━━━━━ 0s 108ms/step - loss: 0.1148 - mae: 0.0999 - mse: 0.0199
Epoch 3: val_loss improved from 0.12181 to 0.11256, saving model to /content/drive/MyDrive/Colorizer/best_colorizer_film.weights.h5
218/218 ━━━━━━━━━━━━━━━━━━━━ 35s 159ms/step - loss: 0.1

TO load an old model pretrained

In [ ]:
model = build_unet_film(img_size=IMG_SIZE, base=32, sep=False)
model.load_weights(CKPT_PATH)

Now we need to teste our model someway

In [25]:
# --- Baseline del prof, invariata ---
# --- Baseline del prof, invariata ---
def baseline(gray, palette):
    m = np.mean(gray, axis=(1,2)) + 1e-12  # evita div/0
    red   = gray * (palette[:,0]/m)[:, np.newaxis, np.newaxis]
    green = gray * (palette[:,1]/m)[:, np.newaxis, np.newaxis]
    blue  = gray * (palette[:,2]/m)[:, np.newaxis, np.newaxis]
    return np.stack([red, green, blue], axis=-1)  # (B,H,W,3)

# --- Valutazione su tutto il test set (no print, solo return) ---
def evaluate_on_test(model, test_gen):
    all_mse_model = []
    all_mse_base  = []
    all_ratio     = []

    for i in range(len(test_gen)):
        # 1) batch dal test
        (L_batch, palette_batch), ab_true = test_gen[i]  # L:(B,H,W,1), palette:(B,3), ab_true:(B,H,W,2)

        # 2) predizione del modello e conversione in RGB
        ab_pred  = model.predict([L_batch, palette_batch], verbose=0)     # (B,H,W,2)
        rgb_true = lab_batch_to_rgb(L_batch, ab_true)                      # (B,H,W,3) in [0,1]
        rgb_pred = lab_batch_to_rgb(L_batch, ab_pred)                      # (B,H,W,3) in [0,1]

        # 3) baseline (usa gray come media dei 3 canali da rgb_true)
        gray_from_true = np.mean(rgb_true, axis=-1)                        # (B,H,W)
        rgb_base = baseline(gray_from_true, palette_batch)                 # (B,H,W,3)

        # 4) MSE per immagine
        mse_model = np.mean((rgb_pred - rgb_true)**2, axis=(1,2,3))        # (B,)
        mse_base  = np.mean((rgb_base - rgb_true)**2, axis=(1,2,3))        # (B,)

        # 5) ratio per immagine (Generated / Initial)
        ratio = mse_model / (mse_base + 1e-12)

        # 6) accumulo
        all_mse_model.append(mse_model)
        all_mse_base.append(mse_base)
        all_ratio.append(ratio)

    # concateno e faccio le medie globali
    all_mse_model = np.concatenate(all_mse_model)
    all_mse_base  = np.concatenate(all_mse_base)
    all_ratio     = np.concatenate(all_ratio)

    return {
        "mse_model_mean": float(np.mean(all_mse_model)),
        "mse_baseline_mean": float(np.mean(all_mse_base)),
        # questo è quello che di solito si usa come "score": media dei ratio per-immagine
        "ratio_mean": float(np.mean(all_ratio)),
        "ratio_median": float(np.median(all_ratio)),
        "n_processed": int(all_ratio.size),
        # lo rendo robusto: se il generator ha .images lo uso, altrimenti stimo da len * batch_size
        "n_expected": int(len(getattr(test_gen, "images", []))) if hasattr(test_gen, "images")
                       else int(len(test_gen) * getattr(test_gen, "batch_size", 1)),
    }


In [26]:
# --- Esecuzione e stampa "pulita" ---
results = evaluate_on_test(model, test_lab)

mse_model = results["mse_model_mean"]
mse_base  = results["mse_baseline_mean"]
ratio     = results["ratio_mean"]   # media dei ratio per-immagine

print(f"MSE_model={mse_model:.6f}")
print(f"MSE_baseline={mse_base:.6f}")
print(f"ratio={ratio:.5f}")

MSE_model=0.006979
MSE_baseline=0.009577
ratio=0.82829


Proceed with the second training of the same model using MSE

In [28]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(3e-5),
    loss=mse_cos_on_ab(),          # MAE + TV è la loss ottimizzata
    metrics=[tf.keras.metrics.MeanAbsoluteError(name='mae'),
             tf.keras.metrics.MeanSquaredError(name='mse')]
)

history = model.fit(
    train_lab,
    validation_data=val_lab,
    epochs=100,
    callbacks=callbacks,
    verbose=1
)


Epoch 1/100
218/218 ━━━━━━━━━━━━━━━━━━━━ 0s 185ms/step - loss: 0.0407 - mae: 0.0790 - mse: 0.0132
Epoch 1: val_loss improved from 0.09322 to 0.03989, saving model to /content/drive/MyDrive/Colorizer/best_colorizer_film.weights.h5
218/218 ━━━━━━━━━━━━━━━━━━━━ 98s 272ms/step - loss: 0.0407 - mae: 0.0790 - mse: 0.0132 - val_loss: 0.0399 - val_mae: 0.0780 - val_mse: 0.0129 - learning_rate: 3.0000e-05
Epoch 2/100
218/218 ━━━━━━━━━━━━━━━━━━━━ 0s 109ms/step - loss: 0.0392 - mae: 0.0772 - mse: 0.0126
Epoch 2: val_loss improved from 0.03989 to 0.03904, saving model to /content/drive/MyDrive/Colorizer/best_colorizer_film.weights.h5
218/218 ━━━━━━━━━━━━━━━━━━━━ 35s 162ms/step - loss: 0.0392 - mae: 0.0772 - mse: 0.0126 - val_loss: 0.0390 - val_mae: 0.0766 - val_mse: 0.0125 - learning_rate: 3.0000e-05
Epoch 3/100
218/218 ━━━━━━━━━━━━━━━━━━━━ 0s 113ms/step - loss: 0.0389 - mae: 0.0761 - mse: 0.0122
Epoch 3: val_loss improved from 0.03904 to 0.03814, saving model to /content/drive/MyDrive/Colorizer/b

In [29]:
# --- Baseline del prof, invariata ---
# --- Baseline del prof, invariata ---
def baseline(gray, palette):
    m = np.mean(gray, axis=(1,2)) + 1e-12  # evita div/0
    red   = gray * (palette[:,0]/m)[:, np.newaxis, np.newaxis]
    green = gray * (palette[:,1]/m)[:, np.newaxis, np.newaxis]
    blue  = gray * (palette[:,2]/m)[:, np.newaxis, np.newaxis]
    return np.stack([red, green, blue], axis=-1)  # (B,H,W,3)

# --- Valutazione su tutto il test set (no print, solo return) ---
def evaluate_on_test(model, test_gen):
    all_mse_model = []
    all_mse_base  = []
    all_ratio     = []

    for i in range(len(test_gen)):
        # 1) batch dal test
        (L_batch, palette_batch), ab_true = test_gen[i]  # L:(B,H,W,1), palette:(B,3), ab_true:(B,H,W,2)

        # 2) predizione del modello e conversione in RGB
        ab_pred  = model.predict([L_batch, palette_batch], verbose=0)     # (B,H,W,2)
        rgb_true = lab_batch_to_rgb(L_batch, ab_true)                      # (B,H,W,3) in [0,1]
        rgb_pred = lab_batch_to_rgb(L_batch, ab_pred)                      # (B,H,W,3) in [0,1]

        # 3) baseline (usa gray come media dei 3 canali da rgb_true)
        gray_from_true = np.mean(rgb_true, axis=-1)                        # (B,H,W)
        rgb_base = baseline(gray_from_true, palette_batch)                 # (B,H,W,3)

        # 4) MSE per immagine
        mse_model = np.mean((rgb_pred - rgb_true)**2, axis=(1,2,3))        # (B,)
        mse_base  = np.mean((rgb_base - rgb_true)**2, axis=(1,2,3))        # (B,)

        # 5) ratio per immagine (Generated / Initial)
        ratio = mse_model / (mse_base + 1e-12)

        # 6) accumulo
        all_mse_model.append(mse_model)
        all_mse_base.append(mse_base)
        all_ratio.append(ratio)

    # concateno e faccio le medie globali
    all_mse_model = np.concatenate(all_mse_model)
    all_mse_base  = np.concatenate(all_mse_base)
    all_ratio     = np.concatenate(all_ratio)

    return {
        "mse_model_mean": float(np.mean(all_mse_model)),
        "mse_baseline_mean": float(np.mean(all_mse_base)),
        # questo è quello che di solito si usa come "score": media dei ratio per-immagine
        "ratio_mean": float(np.mean(all_ratio)),
        "ratio_median": float(np.median(all_ratio)),
        "n_processed": int(all_ratio.size),
        # lo rendo robusto: se il generator ha .images lo uso, altrimenti stimo da len * batch_size
        "n_expected": int(len(getattr(test_gen, "images", []))) if hasattr(test_gen, "images")
                       else int(len(test_gen) * getattr(test_gen, "batch_size", 1)),
    }


In [30]:
# --- Esecuzione e stampa "pulita" ---
results = evaluate_on_test(model, test_lab)

mse_model = results["mse_model_mean"]
mse_base  = results["mse_baseline_mean"]
ratio     = results["ratio_mean"]   # media dei ratio per-immagine

print(f"MSE_model={mse_model:.6f}")
print(f"MSE_baseline={mse_base:.6f}")
print(f"ratio={ratio:.5f}")

MSE_model=0.005799
MSE_baseline=0.009577
ratio=0.70026
